Decode Schizophrenia
decode a language created by the schizophrenia patient( Terry davis) by classifiying what his mood was based on the phrase embeddings


Comp link : https://www.kaggle.com/t/22b34649b6034c31893c64628773b02a

In [1]:
import torch
import pandas as pd
import numpy as np

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# =========================
# LOAD DATA
# =========================

TRAIN_PATH = "/kaggle/input/competitions/decode-schizophrenia/train.pt"
TEST_PATH = "/kaggle/input/competitions/decode-schizophrenia/test.pt"
BASE_CLF_PATH = "/kaggle/input/competitions/decode-schizophrenia/base_classifier.pth"

train_data = torch.load(TRAIN_PATH, map_location="cpu")
test_data = torch.load(TEST_PATH, map_location="cpu")

X_train_torch = train_data[:, :, :-1].reshape(-1, 768).float()
y_train_torch = train_data[:, :, -1].reshape(-1).long()

X_test_torch = test_data.reshape(-1, 768).float()

X_train = X_train_torch.numpy()
y_train = y_train_torch.numpy()
X_test = X_test_torch.numpy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)

# =========================
# TRAIN GATE ONLY
# =========================
# gate label:
# 0 = old class, use frozen classifier
# 1 = class 5
# 2 = class 6

gate_y = np.where(y_train < 5, 0, np.where(y_train == 5, 1, 2))

gate_clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=5000,
        C=1.0,
        solver="lbfgs",
        class_weight="balanced",
        random_state=42,
    )
)

gate_clf.fit(X_train, gate_y)

gate_pred = gate_clf.predict(X_test)

# =========================
# LOAD FROZEN BASE CLASSIFIER
# =========================

base_clf = torch.nn.Linear(in_features=768, out_features=5, bias=True)
base_clf.load_state_dict(torch.load(BASE_CLF_PATH, map_location="cpu"))
base_clf.eval()

with torch.no_grad():
    logits = base_clf(X_test_torch)
    base_pred = torch.softmax(logits, dim=1).argmax(dim=1).numpy()

# =========================
# FINAL PREDICTION
# =========================

preds = []

for g, b in zip(gate_pred, base_pred):
    if g == 0:
        preds.append(int(b))   # frozen classifier predicts 0-4
    elif g == 1:
        preds.append(5)
    else:
        preds.append(6)

submission = pd.DataFrame({
    "id": range(len(preds)),
    "class": preds,
})

submission.to_csv("/kaggle/working/submission_logreg_gate_only.csv", index=False)

display(submission.head())
print(submission["class"].value_counts().sort_index())

X_train: (2473, 768)
y_train: (2473,)
X_test: (700, 768)


,id,class
0,0,3
1,1,3
2,2,2
3,3,6
4,4,5


class
0     77
1     48
2     96
3     95
4     97
5    140
6    147
Name: count, dtype: int64
